# MVP: Vertraute Filme und kulturelle Vielfalt

Das Notebook sucht **fünf Filme**, die zum persönlichen Geschmack passen. Die Liste bleibt ein Allrounder-Angebot: Der Großteil darf vertraut sein, ein Teil erweitert den kulturellen Horizont.

Von fünf Empfehlungen sind deshalb **drei vertraut** und **zwei kulturell neu**. Kulturell neu bedeutet: Das Produktionsland ist nicht USA oder UK, und Englisch ist nicht unter den Sprachen. Das entspricht der Projektregel, dass mindestens 40 Prozent der Liste aus anderen Produktionsländern und Sprachen kommen.

Die Pipeline hat vier Schritte:

1. **Datenbasis und Input:** Wunsch-Genres und drei selbst bewertete Referenzfilme
2. **Vorfilterung:** unvollständige Filme entfernen und jeden Film als vertraut oder kulturell neu markieren
3. **Heuristik:** einen einfachen Score berechnen
4. **Ranking:** drei vertraute und zwei kulturell neue Filme mit Begründung ausgeben

Es wird nur `pandas` verwendet. Jede Zahl im Score lässt sich deshalb direkt nachvollziehen.

## 1. Datenbasis und Input

Die Daten kommen aus `IMDb movies.csv`. Daraus verwenden wir Genre, Produktionsland, Produktionsfirma, Beschreibung, Bewertung und Anzahl der Stimmen.

Der Nutzer gibt zwei Dinge vor:

- **Genres**, zum Beispiel Drama und Thriller
- **drei Referenzfilme mit eigener Bewertung** von 1 bis 10

Diese Werte können direkt im nächsten Feld geändert werden.

In [15]:
import pandas as pd

pd.set_option("display.max_colwidth", 120)

# Wunsch-Genres des Nutzers.
NUTZER_GENRES = ["Drama", "Thriller"]

# Genau drei Referenzfilme mit eigener Bewertung von 1 bis 10.
REFERENZFILME = {
    "Parasite": 9,
    "Oldboy": 8,
    "City of God": 8,
}

ANZAHL_EMPFEHLUNGEN = 5
ANZAHL_KULTURELL_NEU = 2
BEKANNTE_STIMMEN = 40_000

if len(REFERENZFILME) != 3:
    raise ValueError("Bitte genau drei Referenzfilme angeben.")
if not NUTZER_GENRES:
    raise ValueError("Bitte mindestens ein Genre angeben.")
if not 1 <= ANZAHL_KULTURELL_NEU < ANZAHL_EMPFEHLUNGEN:
    raise ValueError("Die Liste soll vertraute und kulturell neue Filme enthalten.")

Zuerst laden wir nur die benötigten Spalten. Leere Textfelder werden durch einen leeren Text ersetzt, damit spätere Vergleiche nicht an fehlenden Werten scheitern.

In [16]:
spalten = [
    "imdb_title_id", "title", "year", "genre", "country", "language",
    "production_company", "description", "avg_vote", "votes",
]

filme = pd.read_csv("IMDb movies.csv", usecols=spalten, low_memory=False)

text_spalten = [
    "title", "genre", "country", "language",
    "production_company", "description",
]
filme[text_spalten] = filme[text_spalten].fillna("")
filme["votes"] = pd.to_numeric(filme["votes"], errors="coerce").fillna(0)

print(f"Filme in der Datenbasis: {len(filme):,}".replace(",", "."))
filme.head(3)

Filme in der Datenbasis: 85.855


,imdb_title_id,title,year,genre,country,language,production_company,description,avg_vote,votes
0,tt0000009,Miss Jerry,1894,Romance,USA,,Alexander Black Photoplays,The adventures of a female reporter in the 1890s.,5.9,154
1,tt0000574,The Story of the Kelly Gang,1906,"Biography, Crime, Drama",Australia,,J. and N. Tait,True story of notorious Australian outlaw Ned Kelly (1855-80).,6.1,589
2,tt0001892,Den sorte drøm,1911,Drama,"Germany, Denmark",,Fotorama,Two men of high rank are both wooing the beautiful and famous equestrian acrobat Stella. While Stella ignores the je...,5.8,188


Die drei Referenzfilme werden im Katalog gesucht. Bei mehreren Filmen mit demselben Titel verwenden wir den Film mit den meisten Stimmen. Diese Filme beschreiben den Geschmack und werden später nicht erneut empfohlen.

In [17]:
def text_teile(text):
    """Macht aus 'Drama, Thriller' die Menge {'drama', 'thriller'}."""
    return {
        teil.strip().casefold()
        for teil in str(text).split(",")
        if teil.strip() and teil.strip().casefold() not in {"none", "nan"}
    }

referenzen = []
for titel, bewertung in REFERENZFILME.items():
    if not 1 <= bewertung <= 10:
        raise ValueError(f"Bewertung für '{titel}' muss zwischen 1 und 10 liegen.")

    treffer = filme[filme["title"].str.casefold() == titel.casefold()]
    treffer = treffer.sort_values("votes", ascending=False)
    if treffer.empty:
        raise ValueError(f"Referenzfilm nicht gefunden: {titel}")

    film = treffer.iloc[0]
    referenzen.append({
        "id": film["imdb_title_id"],
        "titel": film["title"],
        "genres": text_teile(film["genre"]),
        "bewertung": bewertung,
    })

wunsch_genres = {genre.casefold() for genre in NUTZER_GENRES}
referenz_ids = {film["id"] for film in referenzen}

pd.DataFrame(referenzen)[["titel", "bewertung", "genres"]]

,titel,bewertung,genres
0,Parasite,9,"{comedy, drama, thriller}"
1,Oldboy,8,"{action, mystery, drama}"
2,City of God,8,"{drama, crime}"


## 2. Vorfilterung

Zuerst fallen Filme ohne Genre, Produktionsland oder Sprache weg. Auch die drei Referenzfilme werden nicht erneut empfohlen.

Die übrigen Filme werden in zwei Gruppen eingeteilt:

- **Kulturell neu:** kein Produktionsland ist USA oder UK, und Englisch kommt nicht unter den Sprachen vor.
- **Vertraut:** alle anderen Filme, also auch Hollywood- und englischsprachige Produktionen.

Beide Gruppen bleiben im Kandidatenpool. Die spätere Liste mischt sie.

In [18]:
def ist_kulturell_neu(film):
    laender = text_teile(film["country"])
    sprachen = text_teile(film["language"])
    ausserhalb_usa_uk = laender.isdisjoint({"usa", "uk"})
    ohne_englisch = "english" not in sprachen
    return ausserhalb_usa_uk and ohne_englisch

vollstaendig = filme[
    filme["genre"].str.strip().ne("")
    & filme["country"].str.strip().ne("")
    & filme["language"].str.strip().ne("")
    & ~filme["imdb_title_id"].isin(referenz_ids)
].copy()

vollstaendig["kulturell_neu"] = vollstaendig.apply(ist_kulturell_neu, axis=1)
kandidaten = vollstaendig

pd.DataFrame({
    "Gruppe": ["Vertraut", "Kulturell neu", "Kandidaten insgesamt"],
    "Filme": [
        int((~kandidaten["kulturell_neu"]).sum()),
        int(kandidaten["kulturell_neu"].sum()),
        len(kandidaten),
    ],
})

,Gruppe,Filme
0,Vertraut,47915
1,Kulturell neu,36997
2,Kandidaten insgesamt,84912


## 3. Heuristisches Scoring

Für jeden verbliebenen Film wird ein Score berechnet:

`0,4 × Genre-Match + 0,3 × Popularität + 0,2 × Keyword-Bonus`

Die drei Werte liegen jeweils zwischen 0 und 1:

- **Genre-Match:** Zur Hälfte zählt, wie viele Wunsch-Genres der Film trifft. Zur anderen Hälfte zählt die Genre-Ähnlichkeit zu den drei Referenzfilmen, gewichtet mit deren Bewertung.
- **Popularität:** Anzahl der Stimmen geteilt durch 40.000, maximal 1. Ab 40.000 Stimmen gilt ein Film als bekannt; noch höhere Stimmenzahlen erhöhen den Wert nicht weiter.
- **Keyword-Bonus:** Der Wert ist 1, wenn das Wort `independent` in Beschreibung oder Produktionsfirma steht, sonst 0.

Der höchste mögliche Score ist **0,9**, weil die Gewichte zusammen 0,9 ergeben.

In [19]:
def genre_match(genre_text):
    genres = text_teile(genre_text)
    direkte_treffer = len(genres & wunsch_genres) / len(wunsch_genres)

    gewichtete_aehnlichkeit = 0
    gewicht_summe = 0
    for referenz in referenzen:
        gewicht = referenz["bewertung"] / 10
        if referenz["genres"]:
            aehnlichkeit = len(genres & referenz["genres"]) / len(referenz["genres"])
        else:
            aehnlichkeit = 0
        gewichtete_aehnlichkeit += gewicht * aehnlichkeit
        gewicht_summe += gewicht

    return 0.5 * direkte_treffer + 0.5 * (gewichtete_aehnlichkeit / gewicht_summe)

kandidaten["genre_match"] = kandidaten["genre"].apply(genre_match)
kandidaten["popularitaet"] = (kandidaten["votes"] / BEKANNTE_STIMMEN).clip(0, 1)

suchtext = (
    kandidaten["description"].str.casefold()
    + " "
    + kandidaten["production_company"].str.casefold()
)
kandidaten["keyword_bonus"] = suchtext.str.contains(
    r"\bindependent\b", regex=True
).astype(float)

kandidaten["score"] = (
    0.4 * kandidaten["genre_match"]
    + 0.3 * kandidaten["popularitaet"]
    + 0.2 * kandidaten["keyword_bonus"]
)

print(f"Bewertete Kandidaten: {len(kandidaten):,}".replace(",", "."))
kandidaten[["title", "genre_match", "popularitaet", "keyword_bonus", "score"]].head(3)

Bewertete Kandidaten: 84.912


,title,genre_match,popularitaet,keyword_bonus,score
3,Cleopatra,0.443333,0.011150,0.0,0.180678
4,L'Inferno,0.443333,0.055925,0.0,0.194111
5,"From the Manger to the Cross; or, Jesus of Nazareth",0.443333,0.012100,0.0,0.180963


## 4. Ranking und Begründung

Beide Gruppen werden getrennt nach Score sortiert. In die finale Liste kommen:

- die **drei besten vertrauten Filme**
- die **zwei besten kulturell neuen Filme** aus unterschiedlichen Ländern

So bleiben drei Empfehlungen nah am gewohnten Angebot, und zwei Empfehlungen öffnen den Blick auf andere Länder und Sprachen. Anschließend wird die gemeinsame Liste nach Score sortiert.

Die Begründung nennt den Typ, die passenden Genres, den ähnlichsten Referenzfilm, Land, Sprache und Studio.

In [20]:
def naechster_referenzfilm(genre_text):
    genres = text_teile(genre_text)
    return max(
        referenzen,
        key=lambda referenz: (len(genres & referenz["genres"]), referenz["bewertung"]),
    )["titel"]

def beste_filme(pool, anzahl, verschiedene_laender=False):
    pool = (
        pool[pool["genre_match"] > 0]
        .sort_values(["score", "votes"], ascending=False)
    )
    if not verschiedene_laender:
        return pool.head(anzahl)

    gewaehlt = []
    uebrige = []
    gesehene_laender = set()
    for idx, film in pool.iterrows():
        laender = text_teile(film["country"])
        if laender.isdisjoint(gesehene_laender):
            gewaehlt.append(idx)
            gesehene_laender |= laender
        else:
            uebrige.append(idx)
        if len(gewaehlt) == anzahl:
            break

    if len(gewaehlt) < anzahl:
        gewaehlt.extend(uebrige[: anzahl - len(gewaehlt)])
    return pool.loc[gewaehlt]

def begruendung(film):
    gemeinsame = [
        genre for genre in NUTZER_GENRES
        if genre.casefold() in text_teile(film["genre"])
    ]
    genre_text = ", ".join(gemeinsame) if gemeinsame else "keines der Wunsch-Genres"
    typ = "kulturell neu" if film["kulturell_neu"] else "vertraut"
    stimmen = f"{int(film['votes']):,}".replace(",", ".")

    return (
        f"„{film['title']}“ ({film['year']}) ist {typ} und passt zu {genre_text}. "
        f"Genre-nah an „{naechster_referenzfilm(film['genre'])}“. "
        f"Land: {film['country']}. Sprache: {film['language']}. "
        f"Studio: {film['production_company'] or 'keine Angabe'}. "
        f"Stimmen: {stimmen}."
    )

vertraute_filme = beste_filme(
    kandidaten[~kandidaten["kulturell_neu"]],
    ANZAHL_EMPFEHLUNGEN - ANZAHL_KULTURELL_NEU,
)
kulturell_neue_filme = beste_filme(
    kandidaten[kandidaten["kulturell_neu"]],
    ANZAHL_KULTURELL_NEU,
    verschiedene_laender=True,
)

empfehlungen = (
    pd.concat([vertraute_filme, kulturell_neue_filme])
    .sort_values(["score", "votes"], ascending=False)
    .copy()
)
empfehlungen["Begründung"] = empfehlungen.apply(begruendung, axis=1)

anzeige = empfehlungen[[
    "title", "year", "kulturell_neu", "genre", "country", "language",
    "votes", "avg_vote", "genre_match", "popularitaet",
    "keyword_bonus", "score", "Begründung",
]].rename(columns={
    "title": "Titel",
    "year": "Jahr",
    "kulturell_neu": "Kulturell neu",
    "genre": "Genre",
    "country": "Land",
    "language": "Sprache",
    "votes": "Stimmen",
    "avg_vote": "IMDb-Bewertung",
    "genre_match": "Genre-Match",
    "popularitaet": "Popularität",
    "keyword_bonus": "Keyword-Bonus",
    "score": "Score",
})

for spalte in ["Genre-Match", "Popularität", "Keyword-Bonus", "Score"]:
    anzeige[spalte] = anzeige[spalte].round(3)

anzeige.reset_index(drop=True, inplace=True)
anzeige.index = anzeige.index + 1
anzeige

,Titel,Jahr,Kulturell neu,Genre,Land,Sprache,Stimmen,IMDb-Bewertung,Genre-Match,Popularität,Keyword-Bonus,Score,Begründung
1,Nella valle di Elah,2007,False,"Crime, Drama, Mystery",USA,English,68025,7.1,0.577,1.0,1.0,0.731,„Nella valle di Elah“ (2007) ist vertraut und passt zu Drama. Genre-nah an „Oldboy“. Land: USA. Sprache: English. St...
2,A Scanner Darkly - Un oscuro scrutare,2006,False,"Animation, Crime, Drama",USA,English,105125,7.1,0.523,1.0,1.0,0.709,„A Scanner Darkly - Un oscuro scrutare“ (2006) ist vertraut und passt zu Drama. Genre-nah an „City of God“. Land: US...
3,Jerry Maguire,1996,False,"Comedy, Drama, Romance",USA,"English, American Sign Language",238398,7.3,0.503,1.0,1.0,0.701,"„Jerry Maguire“ (1996) ist vertraut und passt zu Drama. Genre-nah an „Parasite“. Land: USA. Sprache: English, Americ..."
4,Kiki - Consegne a domicilio,1989,True,"Animation, Adventure, Drama",Japan,Japanese,118006,7.8,0.443,1.0,1.0,0.677,„Kiki - Consegne a domicilio“ (1989) ist kulturell neu und passt zu Drama. Genre-nah an „Parasite“. Land: Japan. Spr...
5,La regina dei castelli di carta,2009,True,"Crime, Drama, Thriller","Sweden, Denmark, Germany",Swedish,66839,7.3,0.833,1.0,0.0,0.633,"„La regina dei castelli di carta“ (2009) ist kulturell neu und passt zu Drama, Thriller. Genre-nah an „Parasite“. La..."


Das ist das finale Ergebnis für die Oberfläche: fünf Empfehlungen, jeweils mit ausformuliertem Begründungstext.

In [21]:
anzahl_neu = int(empfehlungen["kulturell_neu"].sum())
anzahl = len(empfehlungen)

print(f"Vertraute Filme:      {anzahl - anzahl_neu} von {anzahl}")
print(f"Kulturell neue Filme: {anzahl_neu} von {anzahl}")
print()

for platz, text in enumerate(empfehlungen["Begründung"], start=1):
    print(f"{platz}. {text}\n")

Vertraute Filme:      3 von 5
Kulturell neue Filme: 2 von 5

1. „Nella valle di Elah“ (2007) ist vertraut und passt zu Drama. Genre-nah an „Oldboy“. Land: USA. Sprache: English. Studio: Warner Independent Pictures (WIP). Stimmen: 68.025.

2. „A Scanner Darkly - Un oscuro scrutare“ (2006) ist vertraut und passt zu Drama. Genre-nah an „City of God“. Land: USA. Sprache: English. Studio: Warner Independent Pictures (WIP). Stimmen: 105.125.

3. „Jerry Maguire“ (1996) ist vertraut und passt zu Drama. Genre-nah an „Parasite“. Land: USA. Sprache: English, American Sign Language. Studio: TriStar Pictures. Stimmen: 238.398.

4. „Kiki - Consegne a domicilio“ (1989) ist kulturell neu und passt zu Drama. Genre-nah an „Parasite“. Land: Japan. Sprache: Japanese. Studio: Kiki's Delivery Service Production Committee. Stimmen: 118.006.

5. „La regina dei castelli di carta“ (2009) ist kulturell neu und passt zu Drama, Thriller. Genre-nah an „Parasite“. Land: Sweden, Denmark, Germany. Sprache: Swedish. St

## Grenzen des MVP

Die Mischung ist fest auf drei vertraute und zwei kulturell neue Filme gesetzt. Damit ist die Mindestquote von 40 Prozent sichtbar, aber noch nicht flexibel.

Weitere Grenzen:

- Das Wort `independent` ist nur ein Stichwort. Es erkennt keine unabhängige Filmproduktion zuverlässig.
- Land und Sprache beschreiben kulturelle Neuheit nur grob.
- Die drei Referenzfilme beeinflussen den Score nur über ihre Genres.
- Es gibt noch keinen Test mit echten Nutzerinnen und Nutzern.

Für einen nächsten Schritt könnten Regie, Schauspieler oder die Filmbeschreibung als zusätzliche Ähnlichkeit ergänzt werden.